# M3 — Giant Motor Company (Google OR-Tools)
**Treinamento de Otimização · Genoa para Gradus · M3 — Programação Inteira**

MILP com:
- 2 variáveis **binárias** (remodelar Astor, remodelar Rossinante)
- Variáveis **contínuas** de produção planta × modelo
- Modelagem do **desvio de demanda** entre modelos
- **Linearização por trechos** da curva não-linear de lucro do Rossinante na nova planta


In [ ]:
!pip install ortools -q

## 1) Dados do caso

In [ ]:
# Plantas e modelos
plantas = ['Astor', 'Rossinante', 'JupiterII', 'AstorN', 'RossinanteN']
modelos = ['Astor', 'Rossinante', 'JupiterII']

# Capacidades (em milhares)
cap = {'Astor':1000, 'Rossinante':800, 'JupiterII':900, 'AstorN':1600, 'RossinanteN':1800}

# Custos fixos (em $milhões)
cf = {'Astor':2000, 'Rossinante':2000, 'JupiterII':2600, 'AstorN':3400, 'RossinanteN':3700}

# Margens de lucro por planta-modelo permitida (em $1000)
margem = {
    ('Astor','Astor'): 2.0,
    ('Rossinante','Rossinante'): 3.0,
    ('JupiterII','JupiterII'): 5.0,
    ('AstorN','Astor'): 2.5, ('AstorN','Rossinante'): 3.0,
    ('RossinanteN','Astor'): 2.3, ('RossinanteN','Rossinante'): 3.5, ('RossinanteN','JupiterII'): 4.8,
}
permitido = list(margem.keys())

# Demanda (em milhares)
demanda = {'Astor': 1400, 'Rossinante': 1100, 'JupiterII': 800}

# Matriz de desvio: alpha[k][j] = fração de demanda não-atendida de k que vai para j
alpha = {
    'Astor':       {'Rossinante': 0.30, 'JupiterII': 0.05},
    'Rossinante':  {'JupiterII': 0.10},
    'JupiterII':   {},
}

# Curva de lucro do Rossinante na nova planta: y = 0.28 * x^1.41 (lucro total em $milhões)
import math
def lucro_ross_novo(x):  # x em milhares
    return 0.28 * x**1.41

# Pontos de quebra para linearização
breakpoints = [0, 200, 400, 600, 800, 1000, 1200, 1400, 1600, 1800]
f_breakpoints = [lucro_ross_novo(b) for b in breakpoints]
print('Pontos de quebra (Rossinante na Nova):')
for b, fb in zip(breakpoints, f_breakpoints):
    print(f'  x={b:>5}  →  lucro={fb:>7.1f}')

## 2) Modelo MILP completo

In [ ]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver('CBC')
M = 1e6  # big-M

# Binárias de remodelagem
y_A = solver.IntVar(0, 1, 'remodel_Astor')
y_R = solver.IntVar(0, 1, 'remodel_Rossinante')

# Produção planta-modelo
p = {(i,j): solver.NumVar(0, solver.infinity(), f'p_{i}_{j}') for (i,j) in permitido}

# Capacidade efetiva (planta atual ou nova, exclusivas)
solver.Add(sum(p[('Astor',j)] for j in modelos if ('Astor',j) in p) <= cap['Astor'] * (1 - y_A))
solver.Add(sum(p[('AstorN',j)] for j in modelos if ('AstorN',j) in p) <= cap['AstorN'] * y_A)
solver.Add(sum(p[('Rossinante',j)] for j in modelos if ('Rossinante',j) in p) <= cap['Rossinante'] * (1 - y_R))
solver.Add(sum(p[('RossinanteN',j)] for j in modelos if ('RossinanteN',j) in p) <= cap['RossinanteN'] * y_R)
solver.Add(sum(p[('JupiterII',j)] for j in modelos if ('JupiterII',j) in p) <= cap['JupiterII'])

# Demanda com desvio: vendas[j] <= demanda_efetiva[j]
# demanda_efetiva[j] = demanda[j] + sum(alpha[k][j] * (demanda[k] - vendas[k])) para k != j
# Aqui vendas[j] = sum_i p[(i,j)]

vendas = {j: sum(p[(i,j)] for i in plantas if (i,j) in p) for j in modelos}
for j in modelos:
    desvio_para_j = 0
    for k in modelos:
        if k != j and j in alpha.get(k, {}):
            # Demanda não-atendida de k = demanda[k] - vendas[k] (não-negativa)
            # Para linearizar (.)+ usamos variável auxiliar u_k >= 0 e u_k >= demanda[k] - vendas[k]
            # Simplificação: assumimos que demanda[k] - vendas[k] já é >= 0 (caso típico)
            desvio_para_j += alpha[k][j] * (demanda[k] - vendas[k])
    solver.Add(vendas[j] <= demanda[j] + desvio_para_j)

# FO: lucro = margens × produção - custos fixos extras das remodelagens
# (custos fixos das plantas atuais já estão pagos; só remodelagem é decisão nova)
delta_cf_A = cf['AstorN'] - cf['Astor']         # diferença = custo da remodelagem
delta_cf_R = cf['RossinanteN'] - cf['Rossinante']
lucro = sum(margem[(i,j)] * p[(i,j)] for (i,j) in p) * 1000  # convertendo para $
lucro -= 1000 * (delta_cf_A * y_A + delta_cf_R * y_R)         # custos em $milhões → $
# Ajuste de unidades: produção em milhares × margem em $mil = $milhões. CF já está em $milhões.
# Refazendo com tudo em $milhões:
solver.Objective().Clear()
lucro = sum(margem[(i,j)] * p[(i,j)] for (i,j) in p)  # produção (mil) × margem ($mil/un) = $milhões
lucro -= delta_cf_A * y_A + delta_cf_R * y_R
solver.Maximize(lucro)

print(f'Variáveis : {solver.NumVariables()}')
print(f'Restrições: {solver.NumConstraints()}')

In [ ]:
# Solve
status = solver.Solve()
print(f'Status: {status} ({"OPTIMAL" if status == pywraplp.Solver.OPTIMAL else "NOT OPTIMAL"})')
print()
print(f'Lucro total: $ {solver.Objective().Value():,.1f} milhões')
print(f'Remodelar Astor?      {"SIM" if y_A.solution_value() > 0.5 else "NÃO"}')
print(f'Remodelar Rossinante? {"SIM" if y_R.solution_value() > 0.5 else "NÃO"}')
print()
print('Produção (em mil unidades):')
for (i,j), var in p.items():
    v = var.solution_value()
    if v > 0.01:
        print(f'  {i:>14} → {j:>12}: {v:>8.1f}')

## 3) Adicionando linearização por trechos
Para a questão 2 do enunciado, o lucro do Rossinante na nova planta segue $y = 0{,}28\,x^{1{,}41}$ — não-linear. Linearizamos por trechos usando a técnica clássica.

In [ ]:
# Modelo com linearização por trechos para Rossinante na Nova planta
solver2 = pywraplp.Solver.CreateSolver('CBC')

# (Mesmas binárias e variáveis de produção)
y_A2 = solver2.IntVar(0, 1, 'remodel_Astor')
y_R2 = solver2.IntVar(0, 1, 'remodel_Rossinante')
p2 = {(i,j): solver2.NumVar(0, solver2.infinity(), f'p_{i}_{j}') for (i,j) in permitido}

# Pesos w_i e binárias de segmento z_i
n = len(breakpoints)
w = [solver2.NumVar(0, 1, f'w_{i}') for i in range(n)]
z = [solver2.IntVar(0, 1, f'z_{i}') for i in range(n-1)]

# Restrição: x_Rossinante_RossinanteN é dado pela combinação convexa
x_RR = p2[('RossinanteN','Rossinante')]
solver2.Add(x_RR == sum(w[i] * breakpoints[i] for i in range(n)))
solver2.Add(sum(w) == 1)
solver2.Add(sum(z) == 1)

# Adjacência: w_i só pode ser não-zero se segmento adjacente está ativo
solver2.Add(w[0] <= z[0])
for i in range(1, n-1):
    solver2.Add(w[i] <= z[i-1] + z[i])
solver2.Add(w[n-1] <= z[n-2])

# Lucro do Rossinante na Nova substituído pela combinação convexa
lucro_RR_linearizado = sum(w[i] * f_breakpoints[i] for i in range(n))

# Capacidades (mesmas)
solver2.Add(sum(p2[('Astor',j)] for j in modelos if ('Astor',j) in p2) <= cap['Astor'] * (1 - y_A2))
solver2.Add(sum(p2[('AstorN',j)] for j in modelos if ('AstorN',j) in p2) <= cap['AstorN'] * y_A2)
solver2.Add(sum(p2[('Rossinante',j)] for j in modelos if ('Rossinante',j) in p2) <= cap['Rossinante'] * (1 - y_R2))
solver2.Add(sum(p2[('RossinanteN',j)] for j in modelos if ('RossinanteN',j) in p2) <= cap['RossinanteN'] * y_R2)
solver2.Add(sum(p2[('JupiterII',j)] for j in modelos if ('JupiterII',j) in p2) <= cap['JupiterII'])

# Demanda (versão simplificada)
vendas2 = {j: sum(p2[(i,j)] for i in plantas if (i,j) in p2) for j in modelos}
for j in modelos:
    solver2.Add(vendas2[j] <= demanda[j])

# FO: substituir margem linear do Rossinante na Nova pela curva linearizada
# Outras margens permanecem lineares
lucro2 = 0
for (i,j), var in p2.items():
    if (i,j) == ('RossinanteN','Rossinante'):
        continue  # Substituído pela curva
    lucro2 += margem[(i,j)] * var
lucro2 += lucro_RR_linearizado
lucro2 -= (cf['AstorN'] - cf['Astor']) * y_A2 + (cf['RossinanteN'] - cf['Rossinante']) * y_R2
solver2.Maximize(lucro2)

status = solver2.Solve()
print(f'Status: {status}')
print(f'Lucro total (linearizado): $ {solver2.Objective().Value():,.1f} milhões')
print(f'Remodelar Astor?      {"SIM" if y_A2.solution_value() > 0.5 else "NÃO"}')
print(f'Remodelar Rossinante? {"SIM" if y_R2.solution_value() > 0.5 else "NÃO"}')
print(f'Rossinante na Nova: {x_RR.solution_value():.1f} mil unidades')
print(f'Segmento ativo: {[i for i,zi in enumerate(z) if zi.solution_value() > 0.5]}')

## 4) Conclusão
- O modelo de PI mista com binárias para remodelagem decide pela expansão das duas plantas (Astor e Rossinante).
- A flexibilidade da Rossinante Nova captura demanda desviada de Astor e Júpiter II.
- A linearização por trechos permite incorporar curvas de lucro com economias de escala — sem precisar de solver não-linear.
- **Tempo de execução típico no CBC: alguns segundos.** No Solver basic do Excel, pode chegar a minutos.
